# Exercícios Unidade 10 — Feature Engineering
**Data:** 17/04/2026  
**Nome:** Elisa Rachel Beninca Martins

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

---
## Exercício 1 — Extração de domínio e flag empresarial
Processar lista de e-mails: extrair domínio e criar flag binária que identifica
infraestrutura empresarial (termina em `.com.br`).

In [ ]:
df1 = pd.DataFrame({
    'email': [
        'ana@empresa.com.br',
        'bruno@gmail.com',
        'carlos@outlook.com',
        'diana@corpdomain.com.br',
        'eve@hotmail.com',
        'ti@fatec.sp.gov.br',
        'suporte@techcorp.com.br'
    ]
})

df1['dominio'] = df1['email'].str.split('@').str[1]
df1['flag_empresarial'] = df1['dominio'].str.endswith('.com.br').astype(int)

print(df1.to_string(index=False))

---
## Exercício 2 — Decomposição de timestamps e flag de fim de semana
Extrair mês e dia da semana; flag = 1 se sábado ou domingo.

In [ ]:
timestamps = [
    '2026-04-13 09:30:00',  # segunda
    '2026-04-15 14:00:00',  # quarta
    '2026-04-17 10:00:00',  # sexta
    '2026-04-18 16:30:00',  # sábado
    '2026-04-19 08:00:00',  # domingo
    '2026-04-20 11:00:00',  # segunda
    '2026-04-25 20:00:00',  # sábado
]

df2 = pd.DataFrame({'timestamp': pd.to_datetime(timestamps)})

df2['mes']              = df2['timestamp'].dt.month
df2['dia_semana']       = df2['timestamp'].dt.day_name()
df2['flag_fim_semana']  = (df2['timestamp'].dt.dayofweek >= 5).astype(int)

print(df2.to_string(index=False))

---
## Exercício 3 — Flag de feriado / evento comercial
Verificar se a data de uma transação coincide com Natal, Ano Novo, Tiradentes ou Black Friday.
Flag = 1 permite que o modelo de IA trate o pico como evento esperado, não anomalia.

In [ ]:
FERIADOS = {
    '12-25': 'Natal',
    '01-01': 'Ano Novo',
    '04-21': 'Tiradentes',
    '11-15': 'Proclamacao da Republica',
}

# Black Friday: última sexta de novembro (hardcoded para os anos relevantes)
BLACK_FRIDAYS = {'2026-11-27', '2025-11-28', '2024-11-29'}

transacoes = [
    '2026-12-25',
    '2026-11-27',
    '2026-04-21',
    '2026-06-15',
    '2026-01-01',
    '2026-07-10',
]

df3 = pd.DataFrame({'data_transacao': pd.to_datetime(transacoes)})

def verificar_evento(data):
    chave    = data.strftime('%m-%d')
    data_str = data.strftime('%Y-%m-%d')
    if chave in FERIADOS:
        return FERIADOS[chave]
    if data_str in BLACK_FRIDAYS:
        return 'Black Friday'
    return None

df3['evento']              = df3['data_transacao'].apply(verificar_evento)
df3['flag_evento_especial'] = df3['evento'].notna().astype(int)

print(df3.to_string(index=False))

---
## Exercício 4 — Modelo RFM (Recência, Frequência, Valor Monetário)
Transformar log transacional em perfil único por cliente.

In [ ]:
historico = pd.DataFrame({
    'id_cliente': [1, 1, 2, 2, 2, 3, 1, 3],
    'data':       ['2026-04-20', '2026-03-15', '2026-04-22', '2026-02-10',
                   '2026-01-05', '2026-04-24', '2026-04-24', '2026-03-01'],
    'valor':      [150.0, 200.0, 80.0, 120.0, 90.0, 300.0, 50.0, 180.0]
})

historico['data'] = pd.to_datetime(historico['data'])
data_ref = pd.Timestamp('2026-04-25')

rfm = historico.groupby('id_cliente').agg(
    recencia=        ('data',  lambda x: (data_ref - x.max()).days),
    frequencia=      ('data',  'count'),
    valor_monetario= ('valor', 'sum')
).reset_index()

print("Modelo RFM por cliente:")
print(rfm.to_string(index=False))

---
## Exercício 5 — Delta e evolução percentual de sensor
Monitorar leitura em dois momentos: calcular variação bruta e percentual,
e sinalizar tendência de crescimento ou queda.

In [ ]:
leituras = pd.DataFrame({
    'sensor_id': ['SNSR-01', 'SNSR-01', 'SNSR-02', 'SNSR-02', 'SNSR-03', 'SNSR-03'],
    'momento':   ['antes',   'depois',  'antes',   'depois',  'antes',   'depois'],
    'valor':     [23.5,       31.2,      100.0,     85.0,      50.0,      50.0]
})

pivot = leituras.pivot(index='sensor_id', columns='momento', values='valor').reset_index()

pivot['delta']         = pivot['depois'] - pivot['antes']
pivot['evolucao_pct']  = ((pivot['depois'] - pivot['antes']) / pivot['antes'] * 100).round(2)
pivot['tendencia']     = pivot['delta'].apply(
    lambda d: 'crescimento' if d > 0 else ('queda' if d < 0 else 'estavel')
)

print(pivot.to_string(index=False))

---
## Exercício 6 — Razão de comprometimento (análise de crédito)
Feature = Dívida / Renda. Uma proporção é mais valiosa para a IA do que as
variáveis isoladas: dívida de R$8.000 com renda de R$3.000 (267%) é muito
mais arriscada do que a mesma dívida com renda de R$20.000 (40%).

In [ ]:
df6 = pd.DataFrame({
    'cliente':       ['Alice', 'Bruno', 'Carlos', 'Diana'],
    'divida_total':  [5000,     1200,    8000,     300],
    'renda_mensal':  [3000,     2500,    3000,     1500]
})

df6['razao_comprometimento'] = (df6['divida_total'] / df6['renda_mensal']).round(4)
df6['nivel_risco']           = df6['razao_comprometimento'].apply(
    lambda r: 'ALTO' if r > 0.5 else ('MEDIO' if r > 0.3 else 'BAIXO')
)

print(df6.to_string(index=False))

# A razão comprime dois atributos em um número proporcional:
# sem ela, a IA veria divida=8000 e renda=3000 como valores separados e difíceis de comparar.
# Com razao=2.67 o modelo sabe imediatamente que a dívida é 267% da renda — alto risco.

---
## Exercício 7 — Indicador de omissão (flag de NaN antes do fillna)
Criar flag de ausência ANTES de preencher o NaN, para preservar a informação
de que houve falha de leitura ou recusa do usuário em preencher o campo.

In [ ]:
df7 = pd.DataFrame({
    'nome':     ['Ana',         'Bruno', 'Carlos',       'Diana', 'Eva'],
    'email':    ['ana@ex.com',  None,    'carlos@ex.com', None,   'eva@ex.com'],
    'telefone': [None,          '9999-0001', None,        '9999-0003', None],
    'salario':  [3000.0,        np.nan,  4500.0,          np.nan, 2800.0]
})

# Passo 1: registrar a ausência ANTES de qualquer preenchimento
for col in ['email', 'telefone', 'salario']:
    df7[f'flag_null_{col}'] = df7[col].isna().astype(int)

# Passo 2: preencher lacunas
df7['email']    = df7['email'].fillna('sem_email@desconhecido.com')
df7['telefone'] = df7['telefone'].fillna('0000-0000')
df7['salario']  = df7['salario'].fillna(df7['salario'].median())

print(df7.to_string(index=False))

---
## Exercício 8 — Ranking interno de produtos por volume de vendas
Gerar coluna de ranking para que o modelo entenda a força relativa de cada
item dentro do grupo, em vez de trabalhar com valores absolutos brutos.

In [ ]:
df8 = pd.DataFrame({
    'produto':       ['Notebook', 'Mouse', 'Teclado', 'Monitor', 'Headset', 'Webcam'],
    'volume_vendas': [150,         420,     310,       95,        280,       175]
})

df8['ranking'] = df8['volume_vendas'].rank(ascending=False, method='dense').astype(int)
df8 = df8.sort_values('ranking').reset_index(drop=True)

print(df8.to_string(index=False))

---
## Exercício 9 — Parser de horário em categorias de turno
Transformar dado numérico contínuo (0–23h) em blocos de comportamento humano:
Madrugada, Manhã, Comercial, Noite.

In [ ]:
logs = pd.DataFrame({
    'log_id':  range(1, 13),
    'horario': [0, 3, 5, 6, 9, 11, 12, 14, 17, 18, 21, 23]
})

def classificar_turno(hora):
    if   0 <= hora <  6:  return 'Madrugada'
    elif 6 <= hora < 12:  return 'Manha'
    elif 12 <= hora < 18: return 'Comercial'
    else:                 return 'Noite'

logs['turno'] = logs['horario'].apply(classificar_turno)

print(logs.to_string(index=False))
print("\nDistribuição por turno:")
print(logs['turno'].value_counts().to_string())

---
## Exercício 10 — Delta do Delta (aceleração) com alerta binário
Calcular a diferença entre as últimas três leituras para identificar aceleração
da variável. Alerta = 1 quando o ritmo de crescimento indica tendência exponencial.

In [ ]:
sensor = pd.DataFrame({
    'leitura_id': range(1, 9),
    'valor':      [10, 12, 11, 15, 22, 35, 60, 110]
})

sensor['delta']           = sensor['valor'].diff()
sensor['delta_do_delta']  = sensor['delta'].diff()

# Alerta quando o delta do delta é positivo: o ritmo de crescimento está acelerando
sensor['alerta_exponencial'] = (sensor['delta_do_delta'] > 0).astype(int)

print(sensor.to_string(index=False))
print("\nLeituras com alerta de aceleração exponencial:")
print(sensor[sensor['alerta_exponencial'] == 1].to_string(index=False))